# Эксперимент h=6
Три временных окна, старты в 10:30. Подбор только для h=6. Для остальных горизонтов взяты настройки прогона 0,341. Смесь прогнозов и семейство калибровки подбираются заново.


In [1]:
from pathlib import Path
import json
import gc
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display
DATA = Path('data')
TRAIN_FILE = 'train_solo_track.parquet'
TEST_FILE = 'test_solo_track.parquet'
TRAIN_PATH, TEST_PATH = (DATA / TRAIN_FILE, DATA / TEST_FILE)
if not (TRAIN_PATH.is_file() and TEST_PATH.is_file()) and Path('/kaggle/input').exists():
    train_matches = list(Path('/kaggle/input').rglob(TRAIN_FILE))
    test_matches = list(Path('/kaggle/input').rglob(TEST_FILE))
    if len(train_matches) != 1 or len(test_matches) != 1:
        raise ValueError(f'Укажите TRAIN_PATH и TEST_PATH вручную: train={train_matches}, test={test_matches}')
    TRAIN_PATH, TEST_PATH = (train_matches[0], test_matches[0])
OUT = Path('artifacts/features_1030_windows_v4_h6_profile28/features')
MODEL_DIR = OUT.parent / 'models'
POSTPROCESS_DIR = OUT.parent / 'postprocessing'
for directory in (OUT, MODEL_DIR, POSTPROCESS_DIR):
    directory.mkdir(parents=True, exist_ok=True)
TARGET = 'target_1h'
STATUSES = [f'status_{i}' for i in range(1, 7)]
STEP = pd.Timedelta(minutes=30)
TARGET_SCALE = 100000
MAX_TREES = 4000
EARLY_STOPPING_ROUNDS = 150
WINDOW_OFFSETS = tuple(range(-4, 5))
LAG_MAX_LOOKBACK_HOURS = 24
LAG_MIN_PAIRS = 336
LAG_MIN_CORRELATION = 0.05
FEATURE_PIPELINE_VERSION = '1030_windows_v4'

## Данные и временные границы
Проверяется полная получасовая панель всех маршрутов. Границы задаются до расчёта любых статистик, использующих цель. Словарь корреляционных лагов обучается только до границы train.


In [2]:
history = pd.read_parquet(TRAIN_PATH, columns=['route_id', 'timestamp', TARGET, *STATUSES]).sort_values(['timestamp', 'route_id'])
test = pd.read_parquet(TEST_PATH, columns=['id', 'route_id', 'timestamp'])
times = pd.DatetimeIndex(history.timestamp.unique())
routes = np.sort(history.route_id.unique())
if not (history.notna().all().all() and test.notna().all().all()):
    raise ValueError('В данных есть пропуски')
if not not history.duplicated(['route_id', 'timestamp']).any():
    raise ValueError('Повторяется ключ маршрута и времени')
if not times.equals(pd.date_range(times[0], times[-1], freq=STEP)):
    raise ValueError('Нарушена получасовая сетка')
if not len(history) == len(times) * len(routes):
    raise ValueError('Неполная панель маршрутов')
if not (test.id.is_unique and (not test.duplicated(['route_id', 'timestamp']).any())):
    raise ValueError('Повторы идентификаторов или ключей теста')
end = times[-1]
if not (end.hour, end.minute) == (10, 30):
    raise ValueError('Последняя точка истории должна быть в 10:30')
expected = pd.MultiIndex.from_product([routes, [end + h * STEP for h in range(1, 9)]], names=['route_id', 'timestamp'])
if not expected.difference(pd.MultiIndex.from_frame(test[['route_id', 'timestamp']])).empty:
    raise ValueError('Тест не покрывает все ожидаемые маршруты и горизонты')
if not len(test) == len(expected):
    raise ValueError('Число строк теста отличается от ожидаемого')
values = {name: history[name].to_numpy(dtype=np.float32).reshape(len(times), len(routes)) for name in [TARGET, *STATUSES]}
del history
if not all((np.isfinite(array).all() for array in values.values())):
    raise ValueError('В числовых рядах есть бесконечные значения')
dates = times[(times.hour == 10) & (times.minute == 30) & (times + 8 * STEP <= end)]
if len(dates) <= 28:
    raise ValueError('Недостаточно дат для временного разбиения')
if len(dates) < 63:
    raise ValueError('Недостаточно истории для трёх недель подбора')
holdout_start = dates[-14]
folds = []
for start in (len(dates)-35, len(dates)-28, len(dates)-21):
    folds.append(dict(train=dates[:start-7], val=dates[start-7:start], tuning=dates[start:start+7], test=dates[-14:]))
split_dates = folds[-1]
tune_start = split_dates['tuning'][0]
LAG_VALIDATION_FIT_BEFORE = split_dates['val'][0].normalize()
display(pd.DataFrame([dict(fold=i+1, period=k, first=v[0], last=v[-1], days=len(v)) for i,f in enumerate(folds) for k,v in f.items()]))


,fold,period,first,last,days
0,1,train,2025-07-28 10:30:00,2025-09-19 10:30:00,54
1,1,val,2025-09-20 10:30:00,2025-09-26 10:30:00,7
2,1,tuning,2025-09-27 10:30:00,2025-10-03 10:30:00,7
3,1,test,2025-10-18 10:30:00,2025-10-31 10:30:00,14
4,2,train,2025-07-28 10:30:00,2025-09-26 10:30:00,61
5,2,val,2025-09-27 10:30:00,2025-10-03 10:30:00,7
6,2,tuning,2025-10-04 10:30:00,2025-10-10 10:30:00,7
7,2,test,2025-10-18 10:30:00,2025-10-31 10:30:00,14
8,3,train,2025-07-28 10:30:00,2025-10-03 10:30:00,68
9,3,val,2025-10-04 10:30:00,2025-10-10 10:30:00,7


## Исторические признаки и окна
Используются исходные признаки: уровни цели и статусов, средние за 2/6 часов, изменения за час, статусные суммы и отношения, суточные лаги, отклонение от среднего и отношение текущего уровня к предыдущим дням.

Вокруг каждого лага строится окно ±4 получасовых шага. Для последних значений и сглаженных сигналов доступны только точки не позже собственного старта: положительные сдвиги не создаются. Одинаковые источники не дублируются: например, соседние окна target_back_0/1/2/4 объединяются в target_back_0…8. Для status_latest добавляются back_1…4. Лаги сутки/двое назад и сезонные слоты получают все девять соседей. Исходные центральные признаки сохранены. Эквивалентные change_1h статусов 4-6 заменены на diff_hourly.

Суммы статусов не являются числом уникальных товаров. Названия warehouse_total_load, hub_buffer и hub_gate_bottleneck обозначают формулы, а не подтверждённые физические показатели. Недоступная история остаётся NaN.

Для цели и каждого статуса рассчитываются общая сумма по всем маршрутам в доступный момент и её изменение за час. Соседние исторические значения используют только левую часть окна у старта.


In [3]:
import pyarrow as pa
import pyarrow.parquet as pq
target = values[TARGET]

def past(array, positions):
    positions = np.asarray(positions, dtype=np.int64)
    valid = (positions >= 0) & (positions < len(array))
    result = array[np.clip(positions, 0, len(array) - 1)].astype(np.float32)
    result[~valid] = np.nan
    return result

def average_available(matrices):
    total = np.zeros_like(matrices[0], dtype=np.float32)
    count = np.zeros_like(total)
    for matrix in matrices:
        valid = np.isfinite(matrix)
        total += np.where(valid, matrix, 0)
        count += valid
    return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)

def rolling(array, steps):
    return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)
means = {(name, steps): rolling(array, steps) for name, array in values.items() for steps in (4, 12)}
level24 = rolling(target, 48)
positions = np.arange(len(times))
extra_signals = {}
for name in STATUSES[:3]:
    extra_signals[name + '_hourly'] = values[name] + past(values[name], positions - 1)
extra_signals['warehouse_total_load'] = sum((extra_signals[name + '_hourly'] for name in STATUSES[:3]))
extra_signals['warehouse_processing_rate'] = extra_signals['status_3_hourly'] / (extra_signals['status_1_hourly'] + np.float32(1e-05))
for name in STATUSES[3:]:
    extra_signals[name + '_diff_hourly'] = values[name] - past(values[name], positions - 2)
extra_signals['hub_buffer'] = values['status_5'] - values['status_4']
extra_signals['hub_gate_bottleneck'] = extra_signals['status_4_diff_hourly'] - extra_signals['status_6_diff_hourly']
extra_signals['target_acceleration'] = past(target, positions - 1) - past(target, positions - 2)
hub_signal = extra_signals['status_6_diff_hourly']
extra_signals['target_vs_mean_6h'] = target - means[TARGET, 12]
recent = means[TARGET, 4]
usual = average_available([past(recent, positions - day * 48) for day in range(1, 8)])
extra_signals['morning_level_ratio'] = np.divide(recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)

def offset_name(base, offset):
    if offset == 0:
        return base
    return f"{base}_offset_{('m' if offset < 0 else 'p')}{abs(offset)}"

def add_window(common, base, array, anchor, offsets=WINDOW_OFFSETS):
    for offset in offsets:
        common[offset_name(base, offset)] = past(array, anchor + offset)

def common_features(origins):
    common = {}
    local_target_lags = sorted({k - offset for k in (0, 1, 2, 4) for offset in WINDOW_OFFSETS if k - offset >= 0})
    for lag in local_target_lags:
        common[f'target_back_{lag}'] = past(target, origins - lag)
    causal_offsets = tuple((offset for offset in WINDOW_OFFSETS if offset <= 0))
    for name, array in values.items():
        for steps in (4, 12):
            add_window(common, f'{name}_mean_{steps}', means[name, steps], origins, causal_offsets)
        if name in STATUSES:
            common[f'{name}_latest'] = array[origins]
            for lag in range(1, 5):
                common[f'{name}_back_{lag}'] = past(array, origins - lag)
            if name in STATUSES[:3]:
                change = array - past(array, positions - 2)
                add_window(common, f'{name}_change_1h', change, origins, causal_offsets)
    for name, array in extra_signals.items():
        add_window(common, name, array, origins, causal_offsets)
    for lag in (48, 96):
        add_window(common, f'target_1h_lag_{lag}', target, origins - lag)
    add_window(common, 'warehouse_total_load_lag_48', extra_signals['warehouse_total_load'], origins - 48)
    add_window(common, 'status_6_diff_hourly_lag_48', hub_signal, origins - 48)
    return common
CORRELATION_COLUMNS = ['selected_status6_lag_steps', 'selected_status6_correlation', 'selected_status6_pair_count', 'status6_lag_valid', 'lagged_status6_signal', 'lagged_status6_signal_available']

for name, series in values.items():
    network = series.sum(axis=1, dtype=np.float64).astype(np.float32)[:, None]
    for suffix, signal in [('latest', network), ('change_1h', network - past(network, positions-2))]:
        extra_signals[f'network_{name}_{suffix}'] = np.broadcast_to(signal, target.shape)


## Корреляционный лаг маршрута
На полной получасовой истории train ищется максимальная положительная Pearson-корреляция target(u) с diff_status6(u−L), L=2…48 шагов. Минимум 336 пар и корреляция >0,05 - эвристики, не тест значимости.

Выбранный L фиксирован для маршрута. Для горизонта h нужен сигнал origin+h−L. Помимо центра добавляются соседи ±2 часа, каждый доступен только при собственном источнике ≤origin. У каждого соседа есть флаг доступности. При слабой связи лаг и сигнал NaN. Для финального refit словарь пересчитывается на всей истории; оценочные parquet и модели сохраняются отдельно.

Для выбора лага и его применения из цели и изменения статуса 6 вычитается среднее того же получаса за предыдущие семь суток. Для каждого окна словарь строится заново до начала val.
Исходный лаг статуса 6 и лаг по остаткам сохраняются отдельными группами.


In [4]:
def calculate_dynamic_shoulder(target_values, status6_diff, timestamps, route_ids, *, fit_before, max_lookback_hours=24, min_pairs=336, min_correlation=0.05, route_block_size=128):
    grid = pd.DatetimeIndex(timestamps)
    route_ids = np.asarray(route_ids)
    target_values, status6_diff = (np.asarray(target_values), np.asarray(status6_diff))
    if target_values.shape != status6_diff.shape or target_values.shape != (len(grid), len(route_ids)):
        raise ValueError('Ожидаются два массива [время, маршрут] одной формы')
    if len(grid) == 0 or not grid.equals(pd.date_range(grid[0], periods=len(grid), freq=STEP)):
        raise ValueError('Для выбора лага нужна упорядоченная получасовая сетка без дубликатов')
    if len(np.unique(route_ids)) != len(route_ids):
        raise ValueError('Идентификаторы маршрутов повторяются')
    steps = float(max_lookback_hours) * 2
    if not np.isfinite(steps) or steps < 2 or (not steps.is_integer()):
        raise ValueError('Граница поиска должна задавать целое число получасов, не меньше двух')
    if not isinstance(min_pairs, (int, np.integer)) or min_pairs < 3:
        raise ValueError('min_pairs должен быть целым числом >=3')
    if not 0 <= min_correlation <= 1 or route_block_size < 1:
        raise ValueError('Некорректные параметры выбора корреляции')
    max_lag = int(steps)
    stop = int(grid.searchsorted(pd.Timestamp(fit_before), side='left'))
    anchors = np.arange(max_lag + 2, stop)
    best_corr = np.full(len(route_ids), np.nan)
    best_lag = np.full(len(route_ids), np.nan)
    best_pairs = np.zeros(len(route_ids), dtype=np.int32)
    for start in range(0, len(route_ids), route_block_size):
        finish = min(start + route_block_size, len(route_ids))
        x = target_values[anchors, start:finish].astype(np.float64)
        for lag in range(2, max_lag + 1):
            z = status6_diff[anchors - lag, start:finish].astype(np.float64)
            valid = np.isfinite(x) & np.isfinite(z)
            count = valid.sum(axis=0)
            xx, zz = (np.where(valid, x, 0), np.where(valid, z, 0))
            divisor = np.maximum(count, 1)
            sx, sz = (xx.sum(axis=0), zz.sum(axis=0))
            covariance = (xx * zz).sum(axis=0) - sx * sz / divisor
            vx = np.maximum((xx * xx).sum(axis=0) - sx * sx / divisor, 0)
            vz = np.maximum((zz * zz).sum(axis=0) - sz * sz / divisor, 0)
            denominator = np.sqrt(vx * vz)
            corr = np.divide(covariance, denominator, out=np.full(finish - start, np.nan), where=(count >= min_pairs) & (denominator > 0))
            corr = np.clip(corr, -1, 1)
            previous = best_corr[start:finish]
            better = np.isfinite(corr) & (~np.isfinite(previous) | (corr > previous))
            best_corr[start:finish][better] = corr[better]
            best_lag[start:finish][better] = lag
            best_pairs[start:finish][better] = count[better]
    accepted = np.isfinite(best_corr) & (best_corr > min_correlation)
    return pd.DataFrame({'route_id': route_ids, 'selected_status6_lag_steps': np.where(accepted, best_lag, np.nan), 'selected_status6_correlation': best_corr, 'selected_status6_pair_count': best_pairs, 'status6_lag_valid': accepted.astype(np.int8)})

def correlation_block(frame, h, lag_table, signal_array, prefix, chunk_rows=200000):
    if h not in range(1, 9) or chunk_rows < 1:
        raise ValueError('Некорректный горизонт или размер блока')
    by_route = lag_table.set_index('route_id').reindex(routes)
    route_index = pd.Index(routes)
    lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)
    extra_columns = [offset_name(base, offset) for offset in WINDOW_OFFSETS if offset != 0 for base in ('lagged_status6_signal', 'lagged_status6_signal_available')]
    names = CORRELATION_COLUMNS + extra_columns
    result = np.full((len(frame), len(names)), np.nan, dtype=np.float32)
    for start in range(0, len(frame), chunk_rows):
        finish = min(start + chunk_rows, len(frame))
        block = frame.iloc[start:finish]
        origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))
        route = route_index.get_indexer(block.route_id)
        if (origin < 0).any() or (route < 0).any():
            raise ValueError('Строка не сопоставлена с историей')
        selected = lag[route]
        for j, name in enumerate(CORRELATION_COLUMNS[:4]):
            result[start:finish, j] = by_route[name].to_numpy(dtype=np.float32)[route]
        for offset in WINDOW_OFFSETS:
            source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64) + offset
            available = np.isfinite(selected) & (source >= 0) & (source <= origin) & (source < len(times))
            signal = signal_array[np.clip(source, 0, len(times) - 1), route].astype(np.float32)
            available &= np.isfinite(signal)
            result[start:finish, names.index(offset_name('lagged_status6_signal', offset))] = np.where(available, signal, np.nan)
            result[start:finish, names.index(offset_name('lagged_status6_signal_available', offset))] = available.astype(np.float32)
    names = [prefix + name for name in names]
    frame = frame.drop(columns=names, errors='ignore')
    return pd.concat([frame, pd.DataFrame(result, columns=names, index=frame.index)], axis=1)
def remove_seasonality(series):
    typical = average_available([past(series, positions - 48*d) for d in range(1,8)])
    return series - typical

residual_target = remove_seasonality(target)
residual_signal = remove_seasonality(hub_signal)

def fit_lags(boundary):
    tables = []
    for y, signal, prefix in ((target,hub_signal,''),(residual_target,residual_signal,'residual_')):
        table = calculate_dynamic_shoulder(y,signal,times,routes,fit_before=boundary,max_lookback_hours=LAG_MAX_LOOKBACK_HOURS,min_pairs=LAG_MIN_PAIRS,min_correlation=LAG_MIN_CORRELATION)
        tables.append(table.rename(columns={c:prefix+c for c in table.columns if c != 'route_id'}))
    return tables[0].merge(tables[1],on='route_id',validate='one_to_one')

def add_correlation_features(frame,h,lag_table):
    frame = correlation_block(frame,h,lag_table,hub_signal,'')
    residual_table = lag_table[['route_id',*['residual_'+c for c in CORRELATION_COLUMNS[:4]]]].copy()
    residual_table.columns = ['route_id',*CORRELATION_COLUMNS[:4]]
    return correlation_block(frame,h,residual_table,residual_signal,'residual_')

validation_lag_table = fit_lags(LAG_VALIDATION_FIT_BEFORE)
validation_lag_table.to_csv(OUT.parent / 'status6_lags_validation.csv', index=False)


## Матрицы восьми горизонтов
Каждая историческая строка - маршрут и старт в 10:30 с известной меткой данного горизонта. Сезонные значения берутся относительно прогнозируемого слота, включая ±2 часа. Средние slot_daily7/slot_weekly4 получают отдельное значение для каждого сдвига, а не усреднение внутри окна. Центральные profile28 и overlap используются только постобработкой. Календарь относится к будущему слоту.

Для профилей slot_daily7 и slot_weekly4 вычисляются сдвиг пика в минутах, размах и разность средних правой и левой частей окна. При нескольких равных максимумах используется их средняя позиция. Для календаря остаются день недели, выходной и циклические координаты дня недели.


In [5]:
def horizon_dataset(h, origins):
    slot = origins + h
    dates = times[origins] + h * STEP
    common = common_features(origins)
    frame = pd.DataFrame({'cutoff': times[origins].repeat(len(routes)), 'timestamp': dates.repeat(len(routes)), 'route_id': np.tile(routes, len(origins)), **{name: matrix.ravel() for name, matrix in common.items()}})
    calendar = dict(dayofweek=dates.dayofweek, is_weekend=(dates.dayofweek >= 5).astype(np.int8))
    calendar['weekday_sin'] = np.sin(2 * np.pi * dates.dayofweek / 7)
    calendar['weekday_cos'] = np.cos(2 * np.pi * dates.dayofweek / 7)
    for name, array in calendar.items():
        frame[name] = np.repeat(array, len(routes))
    frame = add_correlation_features(frame, h, validation_lag_table)
    seasonal_columns = {}
    for offset in WINDOW_OFFSETS:
        for day in (1, 2, 7):
            seasonal_columns[offset_name(f'slot_days_back_{day}', offset)] = past(target, slot - day * 48 + offset).ravel()
        for name, days in [('slot_daily7', range(1, 8)), ('slot_weekly4', (7, 14, 21, 28))]:
            seasonal_columns[offset_name(name, offset)] = average_available([past(target, slot - day * 48 + offset) for day in days]).ravel()
    for base in ('slot_daily7','slot_weekly4'):
        window = np.column_stack([seasonal_columns[offset_name(base,k)] for k in WINDOW_OFFSETS])
        valid = np.isfinite(window)
        maximum = np.where(valid, window, -np.inf).max(axis=1)
        minimum = np.where(valid, window, np.inf).min(axis=1)
        peak = valid & (window == maximum[:,None])
        count = peak.sum(axis=1)
        shift = np.divide(peak @ np.asarray(WINDOW_OFFSETS, dtype=float), count, out=np.full(len(window),np.nan), where=count>0)
        seasonal_columns[base+'_peak_minutes'] = shift * 30
        seasonal_columns[base+'_range'] = np.where(count>0, maximum-minimum, np.nan)
        seasonal_columns[base+'_right_minus_left'] = window[:,5:].mean(axis=1)-window[:,:4].mean(axis=1)
    frame = pd.concat([frame, pd.DataFrame(seasonal_columns, index=frame.index)], axis=1)
    profile = np.zeros((len(origins), len(routes)))
    counts = np.zeros_like(profile)
    for day in range(1, 29):
        level = past(level24, origins - day * 48)
        adjusted = past(target, slot - day * 48) * level24[origins] / np.maximum(level, 1)
        profile += np.nan_to_num(adjusted)
        counts += np.isfinite(adjusted)
    frame['profile28'] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()
    if h == 1:
        lower = np.zeros_like(target[origins])
        upper = past(target, origins - 2)
        for position in (origins - 1, origins):
            total = past(target, position)
            lower, upper = (np.maximum(0, total - upper), np.minimum(total, total - lower))
        seasonal = frame[['slot_daily7', 'slot_weekly4']].mean(axis=1).to_numpy()
        frame['overlap'] = np.where((lower <= upper).ravel(), (lower + upper).ravel() / 2 + seasonal / 2, seasonal)
    else:
        frame['overlap'] = np.nan
    raw_target_sources = []
    for lag in range(9):
        raw_target_sources.append((f'target_back_{lag}', -lag, lag in (0, 1, 2, 4)))
    for lag in (48, 96):
        for offset in WINDOW_OFFSETS:
            raw_target_sources.append((offset_name(f'target_1h_lag_{lag}', offset), -lag + offset, offset == 0))
    for day in (1, 2, 7):
        for offset in WINDOW_OFFSETS:
            raw_target_sources.append((offset_name(f'slot_days_back_{day}', offset), h - day * 48 + offset, offset == 0))
    seen, redundant = (set(), [])
    for name, source, central in sorted(raw_target_sources, key=lambda item: not item[2]):
        if source in seen:
            redundant.append(name)
        else:
            seen.add(source)
    frame = frame.drop(columns=redundant)
    numeric = frame.columns.difference(['cutoff', 'timestamp', 'route_id'])
    return frame.astype({name: 'float32' for name in numeric}).copy()

def write_horizon(h):
    available_origins = np.flatnonzero((times.hour == 10) & (times.minute == 30) & (np.arange(len(times)) + h < len(times)))
    path = OUT / f'history_h{h}.parquet'
    frame = horizon_dataset(h, available_origins)
    frame['y'] = target[available_origins + h].ravel()
    if not frame.columns.is_unique or not ((frame.cutoff.dt.hour == 10) & (frame.cutoff.dt.minute == 30)).all():
        raise ValueError('Нарушена схема признаков или время старта')
    frame.to_parquet(path, index=False, compression='zstd')
    features = horizon_dataset(h, np.array([len(times) - 1]))
    requests = test.loc[test.timestamp.eq(end + h * STEP)]
    future = requests.merge(features, on=['route_id', 'timestamp'], how='left', validate='one_to_one')
    if not future.cutoff.eq(end).all() or not future.id.equals(requests.id.reset_index(drop=True)):
        raise ValueError('Ошибка сопоставления тестовых строк')
    future.to_parquet(OUT / f'submission_h{h}.parquet', index=False)
    print(f'h={h}: история {len(frame):,}, submission {len(future):,}, модельных признаков {len(frame.columns) - 6}')
for h in range(1, 9):
    write_horizon(h)
feature_names = list(dict.fromkeys((name for h in range(1, 9) for name in pq.read_schema(OUT / f'history_h{h}.parquet').names)))
FEATURE_GROUPS = {'calendar': ['dayofweek', 'is_weekend', 'weekday_sin', 'weekday_cos'], 'local_statuses': [name for name in feature_names if any((name.startswith(prefix) for prefix in ('status_1_hourly', 'status_2_hourly', 'status_3_hourly', 'warehouse_total_load', 'warehouse_processing_rate'))) and 'lag_48' not in name], 'hub_combinations': [name for name in feature_names if any((name.startswith(prefix) for prefix in ('status_4_diff_hourly', 'status_5_diff_hourly', 'status_6_diff_hourly', 'hub_buffer', 'hub_gate_bottleneck'))) and 'lag_48' not in name], 'historical_lags': [name for name in feature_names if 'lag_48' in name or 'lag_96' in name], 'target_change': [name for name in feature_names if name.startswith('target_acceleration')], 'correlation_lag': [name for name in feature_names if name in CORRELATION_COLUMNS or name.startswith('lagged_status6_signal')], 'seasonal_windows': [name for name in feature_names if name.startswith('slot_')], 'recent_target': [name for name in feature_names if name.startswith('target_back_')], 'recent_statuses': [name for name in feature_names if any((name.startswith(f'status_{i}_back_') or name == f'status_{i}_latest' for i in range(1, 7)))]}

del values, means, level24, extra_signals, positions, recent, usual
FEATURE_GROUPS['network'] = [name for name in feature_names if name.startswith('network_')]
FEATURE_GROUPS['residual_correlation_lag'] = [name for name in feature_names if name.startswith('residual_')]
(OUT.parent / 'feature_pipeline.json').write_text(json.dumps({'version': FEATURE_PIPELINE_VERSION, 'window_offsets_steps': list(WINDOW_OFFSETS), 'training_origin': '10:30', 'validation_lag_fit_before': str(LAG_VALIDATION_FIT_BEFORE), 'splits': {key: [str(period.min()), str(period.max())] for key, period in split_dates.items()}, 'feature_groups': FEATURE_GROUPS}, ensure_ascii=False, indent=2))


C:\Users\artem\AppData\Local\Temp\ipykernel_21496\554008329.py:6: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  calendar = dict(dayofweek=dates.dayofweek, is_weekend=(dates.dayofweek >= 5).astype(np.int8))
C:\Users\artem\AppData\Local\Temp\ipykernel_21496\554008329.py:6: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  calendar = dict(dayofweek=dates.dayofweek, is_weekend=(dates.dayofweek >= 5).astype(np.int8))
C:\Users\artem\AppData\Local\Temp\ipykernel_21496\554008329.py:7: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  calendar['weekday_sin'] = np.sin(2 * np.pi * dates.dayofweek / 7)
C:\Users\artem\AppData\Local\Temp\ipykernel_21496\554008329.py:8: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed 

h=1: история 96,000, submission 1,000, модельных признаков 378


C:\Users\artem\AppData\Local\Temp\ipykernel_21496\554008329.py:6: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  calendar = dict(dayofweek=dates.dayofweek, is_weekend=(dates.dayofweek >= 5).astype(np.int8))
C:\Users\artem\AppData\Local\Temp\ipykernel_21496\554008329.py:6: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  calendar = dict(dayofweek=dates.dayofweek, is_weekend=(dates.dayofweek >= 5).astype(np.int8))
C:\Users\artem\AppData\Local\Temp\ipykernel_21496\554008329.py:7: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  calendar['weekday_sin'] = np.sin(2 * np.pi * dates.dayofweek / 7)
C:\Users\artem\AppData\Local\Temp\ipykernel_21496\554008329.py:8: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed 

h=2: история 96,000, submission 1,000, модельных признаков 380


C:\Users\artem\AppData\Local\Temp\ipykernel_21496\554008329.py:6: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  calendar = dict(dayofweek=dates.dayofweek, is_weekend=(dates.dayofweek >= 5).astype(np.int8))
C:\Users\artem\AppData\Local\Temp\ipykernel_21496\554008329.py:6: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  calendar = dict(dayofweek=dates.dayofweek, is_weekend=(dates.dayofweek >= 5).astype(np.int8))
C:\Users\artem\AppData\Local\Temp\ipykernel_21496\554008329.py:7: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  calendar['weekday_sin'] = np.sin(2 * np.pi * dates.dayofweek / 7)
C:\Users\artem\AppData\Local\Temp\ipykernel_21496\554008329.py:8: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed 

h=3: история 96,000, submission 1,000, модельных признаков 382


... (8 outputs left out)

C:\Users\artem\AppData\Local\Temp\ipykernel_21496\554008329.py:6: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  calendar = dict(dayofweek=dates.dayofweek, is_weekend=(dates.dayofweek >= 5).astype(np.int8))
C:\Users\artem\AppData\Local\Temp\ipykernel_21496\554008329.py:6: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  calendar = dict(dayofweek=dates.dayofweek, is_weekend=(dates.dayofweek >= 5).astype(np.int8))
C:\Users\artem\AppData\Local\Temp\ipykernel_21496\554008329.py:7: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  calendar['weekday_sin'] = np.sin(2 * np.pi * dates.dayofweek / 7)
C:\Users\artem\AppData\Local\Temp\ipykernel_21496\554008329.py:8: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed 

h=8: история 96,000, submission 1,000, модельных признаков 392


C:\Users\artem\AppData\Local\Temp\ipykernel_21496\554008329.py:6: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  calendar = dict(dayofweek=dates.dayofweek, is_weekend=(dates.dayofweek >= 5).astype(np.int8))
C:\Users\artem\AppData\Local\Temp\ipykernel_21496\554008329.py:6: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  calendar = dict(dayofweek=dates.dayofweek, is_weekend=(dates.dayofweek >= 5).astype(np.int8))
C:\Users\artem\AppData\Local\Temp\ipykernel_21496\554008329.py:7: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  calendar['weekday_sin'] = np.sin(2 * np.pi * dates.dayofweek / 7)
C:\Users\artem\AppData\Local\Temp\ipykernel_21496\554008329.py:8: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed 

12072

## Фиксированные модели и сравнение h=6
Для семи горизонтов конфигурации и число деревьев каждого из трёх окон взяты из вывода experiments.ipynb (Public 0,341). Только для h=6 сравниваем прямую модель и поправку к profile28 с подбором по трём неделям tuning. Test используется для отчёта.


In [6]:
from catboost import CatBoostRegressor
from catboost.utils import get_gpu_device_count
if get_gpu_device_count() < 1:
    raise RuntimeError('Для обучения включите GPU в настройках Kaggle Notebook.')
MODEL_DIR = OUT.parent / 'models'
MODEL_DIR.mkdir(exist_ok=True)
print('Начало подбора:', tune_start, 'Начало отложенной проверки:', holdout_start)

def make_model(trees, params):
    return CatBoostRegressor(iterations=trees, **params, loss_function='MAE', eval_metric='MAE', task_type='GPU', devices='0', cat_features=['route_id'], random_seed=42, allow_writing_files=False, verbose=200)

def mean_daily_score(frame, prediction):
    actual = frame.y.to_numpy(dtype=np.float64)
    errors = np.asarray(prediction, dtype=np.float64) - actual
    totals = pd.DataFrame({'cutoff': frame.cutoff.dt.normalize().to_numpy(), 'volume': actual, 'error': errors, 'absolute': np.abs(errors)}).groupby('cutoff').sum()
    denominator = totals.volume.where(totals.volume > 0)
    return ((totals.absolute + totals.error.abs()) / denominator).mean()

def score_row(frame, predictions, split):
    actual = frame.y.to_numpy(dtype=np.float64)
    errors = np.asarray(predictions, dtype=np.float64) - actual
    volume = actual.sum()
    wape = np.abs(errors).sum() / volume if volume > 0 else np.nan
    signed_bias = errors.sum() / volume if volume > 0 else np.nan
    return {'выборка': split, 'даты': frame.cutoff.dt.normalize().nunique(), 'строки': len(frame), 'WAPE': wape, 'Relative Bias': signed_bias, '|Relative Bias|': abs(signed_bias), 'метрика конкурса': wape + abs(signed_bias), 'среднее по дням': mean_daily_score(frame, predictions)}
TARGET_SCALE = 100000

def model_columns(data):
    return data.columns.drop(['timestamp', 'cutoff', 'y', 'profile28', 'overlap']).tolist()
SEARCH_CONFIGS = [{'depth': 6, 'learning_rate': 0.03, 'l2_leaf_reg': 3, 'bootstrap_type': 'No'}, {'depth': 6, 'learning_rate': 0.07, 'l2_leaf_reg': 10, 'bootstrap_type': 'Bayesian', 'bagging_temperature': 1}, {'depth': 8, 'learning_rate': 0.03, 'l2_leaf_reg': 3, 'bootstrap_type': 'No'}, {'depth': 8, 'learning_rate': 0.05, 'l2_leaf_reg': 10, 'bootstrap_type': 'Bayesian', 'bagging_temperature': 1}, {'depth': 10, 'learning_rate': 0.03, 'l2_leaf_reg': 10, 'bootstrap_type': 'No'}, {'depth': 7, 'learning_rate': 0.05, 'l2_leaf_reg': 3, 'bootstrap_type': 'Bayesian', 'bagging_temperature': 0.5}]

def profile_base(frame, mode):
    if mode == 'residual':
        return frame['profile28'].fillna(0).to_numpy(dtype=np.float64)
    return 0.0

def training_target(frame, mode):
    return (frame.y.to_numpy(dtype=np.float64) - profile_base(frame, mode)) / TARGET_SCALE

def restore_volume(scaled_prediction, frame, mode):
    return np.maximum(scaled_prediction * TARGET_SCALE + profile_base(frame, mode), 0)


FIXED_RUN = {
    1: (2, [2836, 2399, 2926]),
    2: (2, [2041, 1285, 2301]),
    3: (6, [1483, 1503, 1961]),
    4: (2, [795, 591, 1220]),
    5: (2, [863, 773, 673]),
    7: (2, [591, 399, 550]),
    8: (6, [892, 637, 329]),
}


Начало подбора: 2025-10-11 10:30:00 Начало отложенной проверки: 2025-10-18 10:30:00


In [7]:
h6_panels = {}
h6_winners = {}
selected = {}
evaluation_tables = {}
validation_parts = []
search_rows = []
transfer_parts = {stop: [] for stop in range(1, len(folds))}
lag_tables = [fit_lags(f['val'][0].normalize()) for f in folds[:-1]] + [validation_lag_table]
for h in range(1,9):
    history_frame = pd.read_parquet(OUT / f'history_h{h}.parquet')
    columns = model_columns(history_frame)
    fold_frames = [add_correlation_features(history_frame,h,t) for t in lag_tables]
    scores = []
    prefix_winners = {}
    if h == 6:
        candidates = [(trial, mode, params) for mode in ('direct', 'residual')
                      for trial, params in enumerate(SEARCH_CONFIGS, 1)]
    else:
        fixed_trial, fixed_trees = FIXED_RUN[h]
        candidates = [(fixed_trial, 'direct', SEARCH_CONFIGS[fixed_trial - 1])]
    for trial, mode, params in candidates:
        parts = []
        for index, (fold, data) in enumerate(zip(folds, fold_frames)):
            masks = {name: data.cutoff.isin(days) for name,days in fold.items()}
            model = make_model(MAX_TREES if h == 6 else fixed_trees[index], params)
            fit_options = dict(early_stopping_rounds=EARLY_STOPPING_ROUNDS, use_best_model=True) if h == 6 else dict(use_best_model=False)
            model.fit(data.loc[masks['train'], columns], training_target(data.loc[masks['train']], mode),
                      eval_set=(data.loc[masks['val'],columns], training_target(data.loc[masks['val']], mode)),
                      **fit_options)
            rows = []
            for split,mask in masks.items():
                subset = data.loc[mask]
                pred = restore_volume(model.predict(subset[columns]), subset, mode)
                rows.append(score_row(subset,pred,split))
                if split == 'tuning' or (index == len(folds)-1 and split == 'test'):
                    meta = subset[['cutoff','timestamp','route_id','y','profile28','overlap']].copy()
                    meta['catboost'] = pred
                    meta['seasonal'] = subset[['slot_daily7','slot_weekly4']].mean(axis=1)
                    meta['horizon'] = h
                    meta['period'] = split
                    parts.append(meta)
            table = pd.DataFrame(rows)
            print(f'h={h}, {mode}, конфигурация {trial}, окно {index+1}, деревьев {model.tree_count_}')
            display(table)
            search_rows.extend([dict(horizon=h,target_mode=mode,trial=trial,fold=index+1,trees=model.tree_count_,**row) for row in rows])
        combined = pd.concat(parts,ignore_index=True)
        oos = combined.loc[combined.period.eq('tuning')]
        for stop in transfer_parts:
            boundary = folds[stop]['tuning'][0]
            earlier = oos.loc[oos.timestamp < boundary.normalize()]
            prefix_score = mean_daily_score(earlier, earlier.catboost)
            if not np.isfinite(prefix_score):
                raise ValueError('Неконечная метрика на ранних неделях')
            if stop not in prefix_winners or prefix_score < prefix_winners[stop][0]:
                panel = oos.loc[oos.cutoff <= folds[stop]['tuning'][-1]].copy()
                panel['trial'] = trial
                panel['target_mode'] = mode
                prefix_winners[stop] = (prefix_score, panel)
        score = mean_daily_score(oos,oos.catboost)
        if not np.isfinite(score):
            raise ValueError("Неконечная метрика подбора")
        if h == 6 and (mode not in h6_winners or score < h6_winners[mode]['tuning_score']):
            h6_winners[mode] = dict(trial=trial, tuning_score=score)
            h6_panels[mode] = combined.copy()
        if not scores or score < min(scores):
            selected[h] = dict(target_mode=mode,params=params.copy(),trees=int(model.tree_count_),trial=trial,tuning_score=float(score))
            evaluation_tables[h] = table.copy()
            best_part = combined
            model.save_model(str(MODEL_DIR / f'catboost_h{h}_validation.cbm'))
        scores.append(score)
    for stop, (_, panel) in prefix_winners.items():
        transfer_parts[stop].append(panel)
    validation_parts.append(best_part)
    pd.DataFrame(search_rows).to_csv(OUT.parent/'hyperparameter_search.csv',index=False)
    (OUT.parent/'selected_models.json').write_text(json.dumps(selected,indent=2))
validation = pd.concat(validation_parts,ignore_index=True)
validation.to_parquet(OUT.parent/'validation_predictions.parquet',index=False)
print('Локальный test оценён до финального переобучения. Эти даты уже использовались при разработке.')

h6_report = []
for mode, panel in h6_panels.items():
    for period, rows in panel.groupby('period'):
        h6_report.append(dict(target_mode=mode, **h6_winners[mode], **score_row(rows, rows.catboost, period)))
h6_comparison = pd.DataFrame(h6_report)
h6_comparison.to_csv(OUT.parent / 'h6_target_comparison.csv', index=False)
display(h6_comparison)


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.2372940	test: 1.3994329	best: 1.3994329 (0)	total: 23.2ms	remaining: 1m 5s
200:	learn: 0.6660130	test: 0.7702851	best: 0.7702851 (200)	total: 2.38s	remaining: 31.3s
400:	learn: 0.6383184	test: 0.7456254	best: 0.7456254 (400)	total: 4.75s	remaining: 28.9s
600:	learn: 0.6208918	test: 0.7393760	best: 0.7393760 (600)	total: 7.08s	remaining: 26.3s
800:	learn: 0.6060059	test: 0.7368414	best: 0.7368414 (800)	total: 9.36s	remaining: 23.8s
1000:	learn: 0.5921946	test: 0.7353457	best: 0.7353457 (1000)	total: 11.6s	remaining: 21.3s
1200:	learn: 0.5796294	test: 0.7341572	best: 0.7341572 (1200)	total: 13.9s	remaining: 18.9s
1400:	learn: 0.5677614	test: 0.7333393	best: 0.7333393 (1400)	total: 16.2s	remaining: 16.6s
1600:	learn: 0.5567152	test: 0.7317196	best: 0.7317196 (1600)	total: 18.5s	remaining: 14.2s
1800:	learn: 0.5461277	test: 0.7308551	best: 0.7307881 (1765)	total: 20.7s	remaining: 11.9s
2000:	learn: 0.5356315	test: 0.7302070	best: 0.7302070 (2000)	total: 23s	remaining: 9.6s
2200

,выборка,даты,строки,WAPE,Relative Bias,|Relative Bias|,метрика конкурса,среднее по дням
0,train,54,54000,0.191158,-0.036090,0.036090,0.227247,0.226888
1,val,7,7000,0.247495,-0.056987,0.056987,0.304482,0.303797
2,tuning,7,7000,0.255874,-0.037740,0.037740,0.293614,0.292966
3,test,14,14000,0.251628,-0.085583,0.085583,0.337211,0.337238


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.2556016	test: 1.4270831	best: 1.4270831 (0)	total: 13.1ms	remaining: 31.4s
200:	learn: 0.6762038	test: 0.7906924	best: 0.7906924 (200)	total: 2.37s	remaining: 26s
400:	learn: 0.6487984	test: 0.7686989	best: 0.7686989 (400)	total: 4.72s	remaining: 23.5s
600:	learn: 0.6326691	test: 0.7638397	best: 0.7638397 (600)	total: 7.08s	remaining: 21.2s
800:	learn: 0.6190968	test: 0.7621238	best: 0.7621238 (800)	total: 9.41s	remaining: 18.8s
1000:	learn: 0.6064524	test: 0.7613400	best: 0.7613400 (1000)	total: 11.8s	remaining: 16.4s
1200:	learn: 0.5946918	test: 0.7603447	best: 0.7603447 (1200)	total: 14.2s	remaining: 14.1s
1400:	learn: 0.5829893	test: 0.7597935	best: 0.7597935 (1400)	total: 16.6s	remaining: 11.8s
1600:	learn: 0.5727592	test: 0.7593330	best: 0.7593330 (1600)	total: 19s	remaining: 9.45s
1800:	learn: 0.5629573	test: 0.7588847	best: 0.7588741 (1795)	total: 21.3s	remaining: 7.06s
2000:	learn: 0.5534319	test: 0.7586925	best: 0.7585767 (1915)	total: 23.5s	remaining: 4.68s
2200:

,выборка,даты,строки,WAPE,Relative Bias,|Relative Bias|,метрика конкурса,среднее по дням
0,train,61,61000,0.202705,-0.036233,0.036233,0.238938,0.238605
1,val,7,7000,0.255842,-0.033928,0.033928,0.289770,0.290173
2,tuning,7,7000,0.256587,-0.037775,0.037775,0.294362,0.293409
3,test,14,14000,0.250304,-0.075188,0.075188,0.325493,0.325411


... (163 outputs left out)

0:	learn: 1.2750604	test: 1.4081155	best: 1.4081155 (0)	total: 16.1ms	remaining: 5.29s
200:	learn: 0.8917597	test: 1.0116782	best: 1.0116782 (200)	total: 3.21s	remaining: 2.04s
328:	learn: 0.8781752	test: 1.0095882	best: 1.0095882 (328)	total: 5.29s	remaining: 0us
bestTest = 1.00958817
bestIteration = 328
h=8, direct, конфигурация 6, окно 3, деревьев 329


,выборка,даты,строки,WAPE,Relative Bias,|Relative Bias|,метрика конкурса,среднее по дням
0,train,68,68000,0.333550,-0.044554,0.044554,0.378104,0.379621
1,val,7,7000,0.343228,-0.016783,0.016783,0.360010,0.369537
2,tuning,7,7000,0.337823,-0.078196,0.078196,0.416019,0.415502
3,test,14,14000,0.339328,-0.076310,0.076310,0.415638,0.414429


Локальный test оценён до финального переобучения. Эти даты уже использовались при разработке.


,target_mode,trial,tuning_score,выборка,даты,строки,WAPE,Relative Bias,|Relative Bias|,метрика конкурса,среднее по дням
0,direct,2,0.395703,test,14,14000,0.348546,-0.046712,0.046712,0.395259,0.399095
1,direct,2,0.395703,tuning,21,21000,0.342589,-0.040073,0.040073,0.382662,0.395703
2,residual,3,0.389874,test,14,14000,0.351717,-0.000530,0.000530,0.352247,0.393256
3,residual,3,0.389874,tuning,21,21000,0.348179,-0.013521,0.013521,0.361700,0.389874


## Перебор смеси и калибровок
Сравниваются шесть вариантов по средней дневной метрике на двух следующих недельных окнах. Для h6 вариант цели и конфигурация в каждом сравнении выбираются по предшествующим неделям. Семь остальных горизонтов используют настройки прежнего эксперимента. Поэтому оценка переноса условна относительно этих фиксированных моделей. Семейство с лучшим средним val подбирается повторно на всех трёх tuning-неделях перед применением к test и submission.


In [8]:
from scipy.optimize import minimize
from sklearn.isotonic import IsotonicRegression
from scipy.optimize import minimize_scalar

def get_raw_full_prediction(frame, weight, use_overlap, profile_weight):
    base = weight * frame.catboost.to_numpy(dtype=np.float64) + (1 - weight) * frame.seasonal.to_numpy(dtype=np.float64)
    first = frame.horizon.to_numpy() == 1
    if use_overlap:
        base[first] = frame.loc[first, 'overlap'].to_numpy()
    later = ~first
    base[later] = (1 - profile_weight) * base[later] + profile_weight * frame.loc[later, 'profile28'].to_numpy()
    return base

def apply_final_calibration(prediction, kind, value, horizon=None):
    prediction = np.asarray(prediction, dtype=np.float64)
    if kind == 'none':
        result = prediction
    elif kind == 'multiplicative':
        result = prediction * value
    elif kind == 'affine':
        result = prediction * value['k'] + value['b']
    elif kind == 'grouped':
        groups = np.select([np.asarray(horizon)==1, np.asarray(horizon)<=4], [0,1], default=2)
        result = prediction * np.asarray(value['factors'])[groups]
    elif kind == 'nonlinear':
        result = np.interp(prediction, value['x'], value['y'])
    elif kind == 'additive':
        result = prediction + value
    else:
        raise ValueError(f'Неизвестная калибровка: {kind}')
    return np.maximum(result, 0)

def calibration_objective(frame, raw):
    actual = frame.y.to_numpy(dtype=np.float64)
    day_codes, unique_days = pd.factorize(frame.cutoff.dt.normalize(), sort=True)
    volumes = np.bincount(day_codes, weights=actual, minlength=len(unique_days))
    valid = volumes > 0
    if not valid.any() or not np.isfinite(raw).all():
        raise ValueError('Некорректные данные для калибровки')

    def objective(kind, value):
        error = apply_final_calibration(raw, kind, value, frame.horizon) - actual
        absolute = np.bincount(day_codes, weights=np.abs(error), minlength=len(unique_days))
        signed = np.bincount(day_codes, weights=error, minlength=len(unique_days))
        return float(np.mean((absolute[valid] + np.abs(signed[valid])) / volumes[valid]))
    return objective

def fit_calibration_value(frame, raw, kind):
    raw = np.asarray(raw, dtype=np.float64)
    objective = calibration_objective(frame, raw)
    if kind == 'affine':
        scale = max(float(frame.y.mean()), 1.0)
        def params(x):
            return dict(k=float(x[0]), b=float(x[1]*scale))
        initial = np.array([1., 0.])
        fit = minimize(lambda x: objective(kind,params(x)), initial, method='Powell',
                       bounds=[(0.,3.),(-2.,2.)], options={'maxiter':40})
        return min([params(initial),params(fit.x)],key=lambda v: objective(kind,v))
    if kind == 'grouped':
        h = frame.horizon.to_numpy()
        factors = []
        total = raw.sum()
        common = frame.y.sum()/total if total>0 else 1.0
        for mask in (h==1, (h>=2)&(h<=4), h>=5):
            volume = raw[mask].sum()
            ratio = frame.y.to_numpy()[mask].sum()/volume if volume>0 else common
            factors.append(float(np.clip(ratio,0,3)))
        options = [dict(factors=np.clip(common + strength*(np.array(factors)-common),0,3).tolist())
                   for strength in (0.,0.25,0.5,0.75,1.)]
        options.append(dict(factors=[1.,1.,1.]))
        return min(options,key=lambda v: objective(kind,v))
    if kind == 'nonlinear':
        cuts = np.unique(np.quantile(raw,np.linspace(0,1,9)[1:-1]))
        bins = np.searchsorted(cuts,raw)
        weights = 1/frame.groupby('cutoff').y.transform('sum').replace(0,np.inf).to_numpy()
        counts = np.bincount(bins,weights=weights)
        keep = counts>0
        x = np.bincount(bins,weights=weights*raw)[keep]/counts[keep]
        y = np.bincount(bins,weights=weights*frame.y.to_numpy())[keep]/counts[keep]
        curve = IsotonicRegression().fit_transform(x,y,sample_weight=counts[keep])
        return dict(x=x.tolist(),y=curve.tolist())
    if kind == 'none':
        return 0.0
    if kind == 'multiplicative':
        lower, upper = (0.0, 3.0)
        bias_value = frame.y.sum() / raw.sum() if raw.sum() > 0 else 1.0
        candidates = [lower, upper, 1.0, float(np.clip(bias_value, lower, upper))]
    elif kind == 'additive':
        scale = max(float(frame.y.mean()), 1.0)
        lower, upper = (-2 * scale, 2 * scale)
        candidates = [lower, upper, 0.0, float(np.clip(np.mean(frame.y.to_numpy() - raw), lower, upper))]
    else:
        raise ValueError(f'Неизвестная калибровка: {kind}')
    grid = np.linspace(lower, upper, 65)
    grid_scores = np.array([objective(kind,x) for x in grid])
    candidates.append(float(grid[grid_scores.argmin()]))
    minima = [i for i in range(len(grid)) if grid_scores[i] <= grid_scores[max(i-1,0)] and grid_scores[i] <= grid_scores[min(i+1,len(grid)-1)]]
    for i in sorted(minima,key=lambda i:grid_scores[i])[:5]:
        interval = (grid[max(i-1,0)], grid[min(i+1,len(grid)-1)])
        result = minimize_scalar(lambda x: objective(kind,x), bounds=interval, method='bounded')
        if result.success:
            candidates.append(float(result.x))
    return min(candidates, key=lambda value: objective(kind, value))

KINDS = ('none','multiplicative','additive','affine','grouped','nonlinear')

def search_calibrations(frame, kinds):
    best = {}
    rows = []
    for weight in np.linspace(0,1,11):
        for overlap in (False, True):
            for profile in np.linspace(0,1,11):
                raw = get_raw_full_prediction(frame,weight,overlap,profile)
                objective = calibration_objective(frame,raw)
                for kind in kinds:
                    value = fit_calibration_value(frame,raw,kind)
                    score = objective(kind,value)
                    config = dict(w=float(weight),overlap=overlap,p_weight=float(profile),
                                  calib_type=kind,calib_val=value,tuning_score=float(score))
                    rows.append(config)
                    if kind not in best or score < best[kind]['tuning_score']:
                        best[kind] = config
    return best, pd.DataFrame(rows)

blend_oos = []
transfer_rows = []
transfer_settings = []
for stop, parts in transfer_parts.items():
    panel = pd.concat(parts,ignore_index=True)
    boundary = folds[stop]['tuning'][0]
    train = panel.loc[panel.cutoff < boundary]
    val = panel.loc[panel.cutoff >= boundary]
    if train.empty or val.empty or train.timestamp.max() >= val.cutoff.min() or val.cutoff.nunique()!=7:
        raise ValueError('Нарушен порядок недель калибровки')
    configs, _ = search_calibrations(train,KINDS)
    prefix_choice = min(configs.values(), key=lambda config: config['tuning_score'])
    out = val[['cutoff','timestamp','route_id','horizon','y']].copy()
    raw = get_raw_full_prediction(val,prefix_choice['w'],prefix_choice['overlap'],prefix_choice['p_weight'])
    out['fe_prediction'] = apply_final_calibration(raw,prefix_choice['calib_type'],prefix_choice['calib_val'],val.horizon)
    out['model_info_through'] = train.timestamp.max()
    blend_oos.append(out)
    for kind, config in configs.items():
        transfer_settings.append(dict(window=stop, models=panel.groupby('horizon').trial.first().to_dict(), target_modes=panel.groupby('horizon').target_mode.first().to_dict(), **config))
        for label, subset in [('train',train),('val',val)]:
            raw = get_raw_full_prediction(subset,config['w'],config['overlap'],config['p_weight'])
            pred = apply_final_calibration(raw,kind,config['calib_val'],subset.horizon)
            raw_score = mean_daily_score(subset,raw)
            transfer_rows.append(dict(window=stop, family=kind, first=subset.cutoff.min(),
                                      last=subset.cutoff.max(), raw_score=raw_score, gain_vs_same_mix=raw_score-mean_daily_score(subset,pred), **score_row(subset,pred,label)))
pd.concat(blend_oos,ignore_index=True).to_parquet(POSTPROCESS_DIR/'fe_forward_predictions.parquet',index=False)
transfer_metrics = pd.DataFrame(transfer_rows)
transfer_metrics.to_csv(POSTPROCESS_DIR/'calibration_transfer.csv',index=False)
display(transfer_metrics)
transfer_summary = transfer_metrics.pivot_table(index='family',columns='выборка',values='среднее по дням').sort_values('val')
transfer_summary.to_csv(POSTPROCESS_DIR/'calibration_transfer_summary.csv')
(POSTPROCESS_DIR/'calibration_transfer_settings.json').write_text(json.dumps(transfer_settings,ensure_ascii=False,indent=2))
display(transfer_summary)
chosen_family = transfer_summary.index[0]
tuning = validation.loc[validation.period.eq('tuning')].copy()
holdout = validation.loc[validation.period.eq('test')].copy()
configs, postprocess_search = search_calibrations(tuning,[chosen_family])
best_config = configs[chosen_family].copy()
best_score = best_config.pop('tuning_score')
submission_calib_val = best_config['calib_val']
postprocess_search.to_csv(POSTPROCESS_DIR/'search.csv',index=False)
print('Выбрано по переносу на val:',best_config)


,window,family,first,last,raw_score,gain_vs_same_mix,выборка,даты,строки,WAPE,Relative Bias,|Relative Bias|,метрика конкурса,среднее по дням
0,1,none,2025-09-27 10:30:00,2025-10-03 10:30:00,0.342074,0.000000,train,7,56000,0.321171,-1.704106e-02,1.704106e-02,0.338212,0.342074
1,1,none,2025-10-04 10:30:00,2025-10-10 10:30:00,0.347879,0.000000,val,7,56000,0.327590,-8.892464e-03,8.892464e-03,0.336482,0.347879
2,1,multiplicative,2025-09-27 10:30:00,2025-10-03 10:30:00,0.347017,0.015841,train,7,56000,0.320526,5.040564e-03,5.040564e-03,0.325567,0.331176
3,1,multiplicative,2025-10-04 10:30:00,2025-10-10 10:30:00,0.347710,-0.000889,val,7,56000,0.327069,1.551853e-02,1.551853e-02,0.342587,0.348599
4,1,additive,2025-09-27 10:30:00,2025-10-03 10:30:00,0.347017,0.015386,train,7,56000,0.320874,5.270139e-03,5.270139e-03,0.326144,0.331631
5,1,additive,2025-10-04 10:30:00,2025-10-10 10:30:00,0.347710,-0.001191,val,7,56000,0.327262,1.597967e-02,1.597967e-02,0.343242,0.348901
6,1,affine,2025-09-27 10:30:00,2025-10-03 10:30:00,0.347017,0.015840,train,7,56000,0.320525,5.028747e-03,5.028747e-03,0.325554,0.331176
7,1,affine,2025-10-04 10:30:00,2025-10-10 10:30:00,0.347710,-0.000883,val,7,56000,0.327067,1.550654e-02,1.550654e-02,0.342574,0.348593
8,1,grouped,2025-09-27 10:30:00,2025-10-03 10:30:00,0.350706,0.018486,train,7,56000,0.320041,2.011341e-09,2.011341e-09,0.320041,0.332220
9,1,grouped,2025-10-04 10:30:00,2025-10-10 10:30:00,0.348662,0.003063,val,7,56000,0.326594,1.120709e-02,1.120709e-02,0.337801,0.345599


выборка,train,val
family,,
grouped,0.336101,0.345101
affine,0.335572,0.346355
multiplicative,0.335572,0.346356
additive,0.335790,0.346641
none,0.343672,0.347871
nonlinear,0.341350,0.349598


Выбрано по переносу на val: {'w': 1.0, 'overlap': False, 'p_weight': 0.30000000000000004, 'calib_type': 'grouped', 'calib_val': {'factors': [1.0435930088955498, 1.0256778335494476, 1.0197194778582321]}}


## Проверка комбинации и отчёт по ошибкам
Test оценён до переобучения. Эти даты уже использовались в проекте.

Ошибки откалиброванной смеси группируются по горизонту, дню недели, квартилю объёма маршрута на раннем train и нулевому факту. У группы с нулевой суммой факта WAPE не определён. Её вклад виден по сумме абсолютных ошибок.


In [9]:
postprocess_metrics = []
validation_postprocessed = validation.copy()
raw_val_pred = get_raw_full_prediction(validation, best_config['w'], best_config['overlap'], best_config['p_weight'])
validation_postprocessed['raw_full_prediction'] = raw_val_pred
validation_postprocessed['y_pred_tuning_calibration'] = apply_final_calibration(raw_val_pred, best_config['calib_type'], submission_calib_val, validation.horizon)
validation_postprocessed.to_parquet(POSTPROCESS_DIR / 'validation_postprocessed.parquet', index=False)
for split, subset in (('tuning', tuning), ('test', holdout)):
    raw = get_raw_full_prediction(subset, best_config['w'], best_config['overlap'], best_config['p_weight'])
    for label, prediction in (('сезонный', subset.seasonal), ('CatBoost', subset.catboost), ('комбинация без калибровки', raw), ('комбинация с tuning-калибровкой', apply_final_calibration(raw, best_config['calib_type'], submission_calib_val, subset.horizon))):
        postprocess_metrics.append({'period': split, **score_row(subset, prediction, label)})
postprocess_metrics = pd.DataFrame(postprocess_metrics)
display(postprocess_metrics)
postprocess_metrics.to_csv(POSTPROCESS_DIR / 'metrics.csv', index=False)
postprocess_metrics.loc[postprocess_metrics.period.eq('test')].to_csv(POSTPROCESS_DIR / 'holdout_metrics.csv', index=False)
diagnostic_rows = []
for label, use_overlap, profile_weight in (('исходная смесь', False, 0.0), ('только замена h1', True, 0.0), ('только профиль', False, best_config['p_weight']), ('выбранная комбинация', best_config['overlap'], best_config['p_weight'])):
    raw = get_raw_full_prediction(tuning, best_config['w'], use_overlap, profile_weight)
    value = submission_calib_val
    row = {'вариант': label, 'вес профиля': profile_weight}
    for period, subset in (('tuning', tuning), ('test', holdout), ('субботы test', holdout.loc[holdout.cutoff.dt.dayofweek.eq(5)])):
        if subset.empty:
            row[period] = np.nan
        else:
            raw = get_raw_full_prediction(subset, best_config['w'], use_overlap, profile_weight)
            row[period] = mean_daily_score(subset, apply_final_calibration(raw, best_config['calib_type'], value, subset.horizon))
    diagnostic_rows.append(row)
postprocessing_diagnostics = pd.DataFrame(diagnostic_rows)
display(postprocessing_diagnostics)
postprocessing_diagnostics.to_csv(POSTPROCESS_DIR / 'overlap_profile_diagnostics.csv', index=False)
(POSTPROCESS_DIR / 'settings.json').write_text(json.dumps({'best_config': best_config, 'tuning_score': best_score, 'calibration_source': 'family_selected_on_forward_val_then_fit_on_tuning', 'test_evaluated_before_refit': True, 'target_scale': TARGET_SCALE}, ensure_ascii=False, indent=2))

errors = validation_postprocessed.loc[validation_postprocessed.period.eq('test')].copy()
errors['weekday'] = errors.cutoff.dt.dayofweek
errors['zero_actual'] = errors.y.eq(0)
route_means = pd.Series(target[times < folds[0]['val'][0].normalize()].mean(axis=0),index=routes)
size = pd.qcut(route_means.rank(method='first'),4,labels=False,duplicates='drop')
errors['route_size_quartile'] = errors.route_id.map(size)
detail = []
for dimension in ['horizon','weekday','zero_actual','route_size_quartile']:
    for key,group in errors.groupby(dimension):
        residual = group.y_pred_tuning_calibration - group.y
        detail.append(dict(dimension=dimension,group=str(key),absolute_error=residual.abs().sum(),signed_error=residual.sum(),**score_row(group,group.y_pred_tuning_calibration,'test')))
error_report = pd.DataFrame(detail)
display(error_report)
error_report.to_csv(POSTPROCESS_DIR/'test_error_report.csv',index=False)

mix_rows = []
for mode, panel in h6_panels.items():
    all_rows = validation.copy()
    keys = ['cutoff', 'route_id']
    mask = all_rows.horizon.eq(6)
    replacement = panel.set_index(keys).catboost.reindex(pd.MultiIndex.from_frame(all_rows.loc[mask, keys]))
    if replacement.isna().any():
        raise ValueError('Прогноз h6 отсутствует для части строк')
    all_rows.loc[mask, 'catboost'] = replacement.to_numpy()
    for period, part in all_rows.groupby('period'):
        raw = get_raw_full_prediction(part, best_config['w'], best_config['overlap'], best_config['p_weight'])
        pred = apply_final_calibration(raw, best_config['calib_type'], submission_calib_val, part.horizon)
        mix_rows.append(dict(h6_mode=mode, **score_row(part, pred, period)))
mix_comparison = pd.DataFrame(mix_rows)
mix_comparison.to_csv(POSTPROCESS_DIR / 'h6_mix_comparison.csv', index=False)
display(mix_comparison)


,period,выборка,даты,строки,WAPE,Relative Bias,|Relative Bias|,метрика конкурса,среднее по дням
0,tuning,сезонный,21,168000,0.358923,6.825472e-04,6.825472e-04,0.359605,0.383667
1,tuning,CatBoost,21,168000,0.322749,-3.543771e-02,3.543771e-02,0.358187,0.358698
2,tuning,комбинация без калибровки,21,168000,0.322411,-2.429940e-02,2.429940e-02,0.346710,0.348791
3,tuning,комбинация с tuning-калибровкой,21,168000,0.323661,2.004681e-08,2.004681e-08,0.323661,0.341119
4,test,сезонный,14,112000,0.354692,1.455218e-02,1.455218e-02,0.369244,0.381694
5,test,CatBoost,14,112000,0.322426,-5.251172e-02,5.251172e-02,0.374938,0.374616
6,test,комбинация без калибровки,14,112000,0.321336,-3.513496e-02,3.513496e-02,0.356471,0.356542
7,test,комбинация с tuning-калибровкой,14,112000,0.321722,-1.109653e-02,1.109653e-02,0.332819,0.341317


C:\Users\artem\AppData\Local\Temp\ipykernel_21496\3383543910.py:20: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  for period, subset in (('tuning', tuning), ('test', holdout), ('субботы test', holdout.loc[holdout.cutoff.dt.dayofweek.eq(5)])):
C:\Users\artem\AppData\Local\Temp\ipykernel_21496\3383543910.py:20: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  for period, subset in (('tuning', tuning), ('test', holdout), ('субботы test', holdout.loc[holdout.cutoff.dt.dayofweek.eq(5)])):
C:\Users\artem\AppData\Local\Temp\ipykernel_21496\3383543910.py:20: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  for period, subset in (('tuning', tuning), ('test', holdout), ('субботы test', holdout.loc[holdout.cutoff.dt.dayofweek.eq(5)])):
C:\Users\artem\AppData\Lo

,вариант,вес профиля,tuning,test,субботы test
0,исходная смесь,0.0,0.344025,0.352237,0.349003
1,только замена h1,0.0,0.347519,0.348972,0.344679
2,только профиль,0.3,0.341119,0.341317,0.331303
3,выбранная комбинация,0.3,0.341119,0.341317,0.331303


C:\Users\artem\AppData\Local\Temp\ipykernel_21496\3383543910.py:33: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  errors['weekday'] = errors.cutoff.dt.dayofweek


,dimension,group,absolute_error,signed_error,выборка,даты,строки,WAPE,Relative Bias,|Relative Bias|,метрика конкурса,среднее по дням
0,horizon,1,1.021850e+09,-1.520634e+08,test,14,14000,0.247658,-0.036854,0.036854,0.284512,0.285713
1,horizon,2,1.281784e+09,-9.006639e+07,test,14,14000,0.310590,-0.021824,0.021824,0.332414,0.337298
2,horizon,3,1.352355e+09,-1.096918e+07,test,14,14000,0.325625,-0.002641,0.002641,0.328266,0.349874
3,horizon,4,1.342167e+09,-5.237311e+06,test,14,14000,0.325098,-0.001269,0.001269,0.326367,0.354307
4,horizon,5,1.343023e+09,2.588710e+07,test,14,14000,0.341141,0.006576,0.006576,0.347716,0.372447
5,horizon,6,1.363349e+09,1.169160e+08,test,14,14000,0.354692,0.030417,0.030417,0.385109,0.407435
6,horizon,7,1.343271e+09,-1.058816e+08,test,14,14000,0.336980,-0.026562,0.026562,0.363542,0.374182
7,horizon,8,1.375644e+09,-1.381002e+08,test,14,14000,0.335732,-0.033704,0.033704,0.369436,0.385518
8,weekday,0,1.475442e+09,-1.238095e+08,test,2,16000,0.323339,-0.027132,0.027132,0.350471,0.350449
9,weekday,1,1.479106e+09,-6.979011e+07,test,2,16000,0.318552,-0.015031,0.015031,0.333582,0.333603


,h6_mode,выборка,даты,строки,WAPE,Relative Bias,|Relative Bias|,метрика конкурса,среднее по дням
0,direct,test,14,112000,0.320806,-1.500751e-02,1.500751e-02,0.335814,0.341177
1,direct,tuning,21,168000,0.322667,-2.310172e-03,2.310172e-03,0.324977,0.340344
2,residual,test,14,112000,0.321722,-1.109653e-02,1.109653e-02,0.332819,0.341317
3,residual,tuning,21,168000,0.323661,2.004681e-08,2.004681e-08,0.323661,0.341119


## Финальное обучение на всей размеченной истории со стартами 10:30
Словарь лагов пересчитывается на полной получасовой истории. Обучение каждой лучшей конфигурации использует все дни 10:30 с известной меткой своего горизонта, включая бывшие tuning, val и test. Число деревьев фиксировано предыдущим этапом. Сохранённые оценки train/test относятся к моделям до refit. Коэффициент калибровки остаётся с tuning; перенос его на переобученную модель - допущение, а требует проверки сохранения bias.

Оценка test независима от финального переобучения и получена до него. При интерпретации учитывайте предыдущие эксперименты на этих датах.


In [10]:
final_lag_table = fit_lags(end + STEP)
final_lag_table.to_csv(OUT.parent / 'status6_lags_final.csv', index=False)
metadata_path = OUT.parent / 'feature_pipeline.json'
metadata = json.loads(metadata_path.read_text())
metadata.update(final_lag_fit_before=str(end + STEP), target_scale=TARGET_SCALE)
metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2))
submission_parts = []
for h in range(1, 9):
    data = add_correlation_features(pd.read_parquet(OUT / f'history_h{h}.parquet'), h, final_lag_table)
    future = add_correlation_features(pd.read_parquet(OUT / f'submission_h{h}.parquet'), h, final_lag_table)
    columns = model_columns(data)
    if not ((data.cutoff.dt.hour == 10) & (data.cutoff.dt.minute == 30)).all():
        raise ValueError('Финальный train содержит старты не в 10:30')
    model = make_model(selected[h]['trees'], selected[h]['params'])
    mode = selected[h]['target_mode']
    model.fit(data[columns], training_target(data, mode))
    model.save_model(str(MODEL_DIR / f'catboost_h{h}_scaled.cbm'))
    result = future[['id', 'profile28', 'overlap']].copy()
    result['seasonal'] = future[['slot_daily7', 'slot_weekly4']].mean(axis=1)
    result['horizon'] = h
    result['catboost'] = restore_volume(model.predict(future[columns]), future, mode)
    full_pred = restore_volume(model.predict(data[columns]), data, mode)
    full_row = score_row(data, full_pred, 'вся история 10:30 после переобучения')
    print(f'h={h}: локальная оценка ранее просмотренного test получена до переобучения; последняя строка — in-sample')
    display(pd.concat([evaluation_tables[h], pd.DataFrame([full_row])], ignore_index=True))
    submission_parts.append(result)
    del model, data, future, full_pred
    gc.collect()

Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.3132392	total: 13.2ms	remaining: 38.6s
200:	learn: 0.7104467	total: 2.52s	remaining: 34.2s
400:	learn: 0.6845505	total: 4.96s	remaining: 31.2s
600:	learn: 0.6715774	total: 7.41s	remaining: 28.7s
800:	learn: 0.6614813	total: 9.88s	remaining: 26.2s
1000:	learn: 0.6524663	total: 12.3s	remaining: 23.7s
1200:	learn: 0.6441093	total: 14.8s	remaining: 21.3s
1400:	learn: 0.6363774	total: 17.3s	remaining: 18.8s
1600:	learn: 0.6290312	total: 20s	remaining: 16.6s
1800:	learn: 0.6216731	total: 22.8s	remaining: 14.2s
2000:	learn: 0.6145851	total: 25.5s	remaining: 11.8s
2200:	learn: 0.6080544	total: 28.4s	remaining: 9.35s
2400:	learn: 0.6015227	total: 30.8s	remaining: 6.74s
2600:	learn: 0.5950627	total: 33.3s	remaining: 4.16s
2800:	learn: 0.5887903	total: 36.2s	remaining: 1.61s
2925:	learn: 0.5850695	total: 37.7s	remaining: 0us
h=1: локальная оценка ранее просмотренного test получена до переобучения; последняя строка — in-sample


,выборка,даты,строки,WAPE,Relative Bias,|Relative Bias|,метрика конкурса,среднее по дням
0,train,68,68000,0.199945,-0.035967,0.035967,0.235911,0.235633
1,val,7,7000,0.256030,-0.033395,0.033395,0.289425,0.288561
2,tuning,7,7000,0.244703,-0.049950,0.049950,0.294653,0.294290
3,test,14,14000,0.249933,-0.077087,0.077087,0.327019,0.326987
4,вся история 10:30 после переобучения,96,96000,0.212400,-0.036695,0.036695,0.249095,0.248762


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.3169197	total: 15.1ms	remaining: 34.8s
200:	learn: 0.8734373	total: 3s	remaining: 31.3s
400:	learn: 0.8568048	total: 5.97s	remaining: 28.3s
600:	learn: 0.8453427	total: 8.95s	remaining: 25.3s
800:	learn: 0.8356529	total: 12s	remaining: 22.6s
1000:	learn: 0.8265877	total: 15.3s	remaining: 19.8s
1200:	learn: 0.8183026	total: 18s	remaining: 16.5s
1400:	learn: 0.8102406	total: 20.5s	remaining: 13.2s
1600:	learn: 0.8025824	total: 23.4s	remaining: 10.3s
1800:	learn: 0.7952284	total: 26s	remaining: 7.22s
2000:	learn: 0.7878889	total: 29.5s	remaining: 4.43s
2200:	learn: 0.7809372	total: 32.2s	remaining: 1.46s
2300:	learn: 0.7774675	total: 33.5s	remaining: 0us
h=2: локальная оценка ранее просмотренного test получена до переобучения; последняя строка — in-sample


,выборка,даты,строки,WAPE,Relative Bias,|Relative Bias|,метрика конкурса,среднее по дням
0,train,68,68000,0.271261,-0.035678,0.035678,0.306939,0.307108
1,val,7,7000,0.313947,-0.029116,0.029116,0.343063,0.342553
2,tuning,7,7000,0.307751,-0.045250,0.045250,0.353002,0.352743
3,test,14,14000,0.310744,-0.068877,0.068877,0.379622,0.379245
4,вся история 10:30 после переобучения,96,96000,0.281477,-0.036679,0.036679,0.318156,0.318306


... (14 outputs left out)

,выборка,даты,строки,WAPE,Relative Bias,|Relative Bias|,метрика конкурса,среднее по дням
0,train,68,68000,0.334905,-0.040955,0.040955,0.375860,0.377661
1,val,7,7000,0.341521,-0.031914,0.031914,0.373435,0.380890
2,tuning,7,7000,0.342386,-0.063116,0.063116,0.405501,0.407398
3,test,14,14000,0.340465,-0.070571,0.070571,0.411036,0.409750
4,вся история 10:30 после переобучения,96,96000,0.336404,-0.041500,0.041500,0.377904,0.379138


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.3129624	total: 17.8ms	remaining: 5.85s
200:	learn: 0.9223174	total: 3.35s	remaining: 2.13s
328:	learn: 0.9106643	total: 5.47s	remaining: 0us
h=8: локальная оценка ранее просмотренного test получена до переобучения; последняя строка — in-sample


,выборка,даты,строки,WAPE,Relative Bias,|Relative Bias|,метрика конкурса,среднее по дням
0,train,68,68000,0.333550,-0.044554,0.044554,0.378104,0.379621
1,val,7,7000,0.343228,-0.016783,0.016783,0.360010,0.369537
2,tuning,7,7000,0.337823,-0.078196,0.078196,0.416019,0.415502
3,test,14,14000,0.339328,-0.076310,0.076310,0.415638,0.414429
4,вся история 10:30 после переобучения,96,96000,0.334846,-0.044059,0.044059,0.378905,0.380175


## Сабмит
Комбинация и калибровка применяются один раз после объединения горизонтов. Параметры и коэффициент взяты только с tuning. Порядок id сохраняется из конкурсного test.


In [11]:
predictions = pd.concat(submission_parts, ignore_index=True)
raw_submission_pred = get_raw_full_prediction(predictions, best_config['w'], best_config['overlap'], best_config['p_weight'])
predictions['y_pred'] = apply_final_calibration(raw_submission_pred, best_config['calib_type'], submission_calib_val, predictions.horizon)
submission = test[['id']].merge(predictions[['id', 'y_pred']], on='id', how='left', validate='one_to_one')
if len(predictions) != len(test) or len(submission) != len(test) or (not predictions.id.is_unique) or (set(predictions.horizon.unique()) != set(range(1, 9))) or (not submission.id.equals(test.id.reset_index(drop=True))) or (not np.isfinite(submission.y_pred).all()) or (submission.y_pred < 0).any():
    raise ValueError('Неполный или некорректный submission')
submission.to_csv('submission.csv', index=False)
print(f'submission.csv: {len(submission):,} строк. Локальная оценка ранее просмотренного test получена до финального переобучения.')
display(submission.head())


submission.csv: 8,000 строк. Локальная оценка ранее просмотренного test получена до финального переобучения.


,id,y_pred
0,0,76004.842882
1,1,150226.899161
2,2,169128.651166
3,3,159566.374353
4,4,159494.610417
